# 24. 임계값 튜닝 (Threshold Tuning)

목적: `val_calib` 데이터를 사용하여 지정된 False Positive Rate (FPR) 허용치를 만족하는 최적의 임계값(Threshold)을 탐색합니다.

In [ ]:
import sys, os, json, joblib
import numpy as np, pandas as pd
sys.path.insert(0, os.path.abspath("..\\"))

from src.eval_core import ThresholdTuner
import config.eval_config as cfg
import config.train_config as tcfg
print("환경 준비 완료")

## 1. 학습된 앙상블 로드


In [ ]:
from pathlib import Path
SAVE_DIR = Path(tcfg.MODEL_SAVE_DIR)

with open(SAVE_DIR / "feature_cols.json", encoding="utf-8") as f:
    FEATURE_COLS = json.load(f)

models = [joblib.load(p) for p in sorted(SAVE_DIR.glob("subset_*.pkl"))]
print(f"로드: {len(models)}개 서브셋 모델, 피처: {len(FEATURE_COLS)}개")

class _EnsembleInfer:
    def __init__(self, models): self.models = models
    def predict_proba(self, df, feature_cols):
        return np.mean([m.predict_proba(df[feature_cols])[:,1] for m in self.models], axis=0)

ensemble = _EnsembleInfer(models)

## 2. 임계값 튜닝 (val_calib)


In [ ]:
df_calib = pd.read_parquet(cfg.VAL_CALIB_PATH)
y_calib_prob = ensemble.predict_proba(df_calib, FEATURE_COLS)

tuner = ThresholdTuner(max_fpr=cfg.MAX_FPR, n_grid=cfg.THRESHOLD_N_GRID)
tuner_result = tuner.fit(df_calib[cfg.TARGET_COL].values, y_calib_prob)
tuner.plot()

THRESHOLD = float(tuner.best_threshold)
print(f"최적 임계값: {THRESHOLD:.4f}")

with open(SAVE_DIR / "best_threshold.json", "w", encoding="utf-8") as f:
    json.dump({"threshold": THRESHOLD}, f)
print("최적 임계값이 저장되었습니다.")